# 🏥 SCOGS — MedGemma 27B Extraction Run (Colab, A100, Ollama)

Single-stage extraction run. The model is loaded from Google Drive if available,
otherwise built from the `.gguf` and cached to Drive for future runs.

| Step | What |
|:--|:--|
| 1 | Verify the GPU |
| 2 | Repository and dependencies |
| 3 | Unit tests |
| 4 | Model setup (load from Drive or build + cache) |
| 5 | Preflight gate |
| 6 | Run the extraction |
| 7 | Results |
| 8 | Hand-check worksheet |
| 9 | Absence audit |
| 10 | Save artifacts |

## Step 1 — Verify the GPU

In [ ]:
!nvidia-smi

import psutil

try:
    import torch
    assert torch.cuda.is_available(), (
        "CUDA GPU not detected. Runtime -> Change runtime type -> A100 GPU.")
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except ImportError:
    device_name, vram_gb = "(torch unavailable - see nvidia-smi above)", 0.0

sys_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"\n[GPU]        {device_name}")
print(f"[VRAM]       {vram_gb:.1f} GB")
print(f"[System RAM] {sys_ram_gb:.1f} GB")

## Step 2 — Repository and dependencies

Only installs what is actually missing. The repo is cloned from GitHub and checked
out to the branch — uncommitted local fixes are invisible here.

In [ ]:
import importlib.util, os, subprocess, sys

BRANCH = "p11-harness-measurement-fixes"      # <- the branch to run from

REPO = "/content/st_jude"
if not os.path.exists(REPO):
    !git clone --quiet https://github.com/Edward-Bae-00/st_jude.git {REPO}
%cd {REPO}
!rm -rf {REPO}/st_jude
!find {REPO} -name __pycache__ -exec rm -rf {} + 2>/dev/null
!git fetch --quiet origin && git checkout --quiet {BRANCH} && git pull --quiet origin {BRANCH}
print(f"on branch {BRANCH}, HEAD = {subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()}")

# Dependencies
for pkg, pip in [("pandas", "pandas"), ("pint", "pint")]:
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pip}

sys.path.insert(0, f"{REPO}/scripts/experiments")
sys.path.insert(0, f"{REPO}/scripts/scogs")

from medgemma_extraction import unit_guard, reconcile, harness_status
print("harness OK: unit_guard, reconcile, harness_status all importable.")

## Step 3 — Unit tests

In [ ]:
!pytest -q

## Step 4 — Model setup

Loads the model into Ollama's local store. Priority order:
1. Local store already has it (blobs verified) → skip
2. Drive has a complete Ollama store → restore it
3. Drive has a merged `.gguf` → `ollama create`
4. Drive has split GGUF shards → merge, save to Drive, `ollama create`

After a successful build, the Ollama store is **always** saved back to Drive
so future runs skip straight to a restore.

In [ ]:
import json, os, pathlib, shutil, subprocess, time, urllib.error, urllib.request

# ═══════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════
MODEL = "medgemma-27b-bf16"
MODEL_GB = 54.0

# ═══════════════════════════════════════════════════════════════════════
# VRAM / parallelism
# ═══════════════════════════════════════════════════════════════════════
try:
    import torch
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except Exception:
    VRAM_GB = 0.0

NUM_CTX = 4096
KV_GB_PER_SLOT = 2.0
headroom_gb = max(0.0, VRAM_GB - MODEL_GB - 4.0)
NUM_PARALLEL = max(1, min(4, int(headroom_gb // KV_GB_PER_SLOT)))
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['OLLAMA_NUM_PARALLEL'] = str(NUM_PARALLEL)
os.environ['OLLAMA_CONTEXT_LENGTH'] = str(NUM_CTX)
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
print(f"VRAM {VRAM_GB:.1f} GB  model ~{MODEL_GB} GB  parallel={NUM_PARALLEL}")

# ═══════════════════════════════════════════════════════════════════════
# Find the best local disk for serving
# ═══════════════════════════════════════════════════════════════════════
_SKIP_FS = {"tmpfs", "devtmpfs", "squashfs", "proc", "sysfs", "cgroup", "cgroup2",
            "devpts", "fuse", "fuse.drive", "fuseblk"}
_disks = {}
for _line in open("/proc/mounts"):
    _p = _line.split()
    if len(_p) < 3: continue
    _target, _fs = _p[1], _p[2]
    if _fs in _SKIP_FS or "/drive" in _target: continue
    try: _u = shutil.disk_usage(_target)
    except OSError: continue
    if os.access(_target, os.W_OK) and _u.total > 20e9:
        _disks[_target] = _u.free

_ranked = sorted(_disks.items(), key=lambda kv: -kv[1])
_root = _ranked[0][0] if _ranked else "/root"
SERVE_DIR = (pathlib.Path(_root) / "ollama_models" if _root != "/"
             else pathlib.Path("/root/.ollama/models"))
SERVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['OLLAMA_MODELS'] = str(SERVE_DIR)
print(f"serving from {SERVE_DIR}  ({shutil.disk_usage(SERVE_DIR).free / 1e9:.0f} GB free)")

# ═══════════════════════════════════════════════════════════════════════
# Helpers
# ═══════════════════════════════════════════════════════════════════════
HOST = "http://localhost:11434"
OLLAMA_LOG = pathlib.Path("/content/ollama.log")

def manifest_path(root, model):
    name, _, tag = model.partition(":")
    parts = name.split("/")
    if len(parts) == 1:   parts = ["registry.ollama.ai", "library"] + parts
    elif len(parts) == 2: parts = ["registry.ollama.ai"] + parts
    return pathlib.Path(root, "manifests", *parts, tag or "latest")

def store_report(root, model):
    man = manifest_path(root, model)
    if not man.is_file(): return False, [f"no manifest at {man}"]
    try: doc = json.loads(man.read_text())
    except (OSError, ValueError) as e: return False, [f"manifest unreadable: {e}"]
    named = [(l["digest"], l.get("size"))
             for l in [doc.get("config") or {}] + list(doc.get("layers") or [])
             if l.get("digest")]
    if not named: return False, [f"manifest names no blobs"]
    problems = []
    for digest, size in named:
        blob = pathlib.Path(root, "blobs", digest.replace(":", "-"))
        if not blob.exists():
            problems.append(f"missing blob {digest}")
        elif size and blob.stat().st_size != size:
            problems.append(f"short blob {digest}")
    return not problems, problems

def store_intact(root, model): return store_report(root, model)[0]

def daemon_up(host=HOST, timeout=2):
    try: urllib.request.urlopen(host, timeout=timeout); return True
    except Exception: return False

def sh(*cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} failed ({r.returncode}):\n{(r.stderr or r.stdout)[-1500:]}")
    return r.stdout

def sh_stream(*cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    while True:
        chunk = p.stdout.read(256)
        if not chunk: break
        print(chunk, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{cmd[0]} failed ({p.returncode})")

def can_generate(tag, timeout=900):
    body = json.dumps({"model": tag, "prompt": "ok", "stream": False,
                       "options": {"num_predict": 1}}).encode("utf-8")
    req = urllib.request.Request(f"{HOST}/api/generate", data=body,
                                 headers={"Content-Type": "application/json"})
    try:
        urllib.request.urlopen(req, timeout=timeout); return True, ""
    except urllib.error.HTTPError as e:
        return False, f"HTTP {e.code} - {e.read().decode('utf-8', 'replace')[:300]}"
    except Exception as e:
        return False, str(e)

def gguf_complete(q, want_bytes):
    try:
        if q is None or not q.exists() or q.stat().st_size < want_bytes * 0.99: return False
        with open(q, "rb") as fh: return fh.read(4) == b"GGUF"
    except OSError: return False

def tree_bytes(d):
    return sum(q.stat().st_size for q in pathlib.Path(d).rglob('*') if q.is_file())

# ═══════════════════════════════════════════════════════════════════════
# Mount Drive
# ═══════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/scogs_ollama_models')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# Detect what's available
# ═══════════════════════════════════════════════════════════════════════
DRIVE_STORE_OK, _ = store_report(DRIVE_DIR, MODEL)
_singles = [q for q in DRIVE_DIR.glob("**/*.gguf") if "-of-" not in q.name]
MERGED_IN_DRIVE = max(_singles, key=lambda q: q.stat().st_size) if _singles else None
_heads = sorted(DRIVE_DIR.glob("**/*-00001-of-*.gguf"))
SHARD_HEAD = _heads[0] if _heads else None
N_SHARDS = len(list(SHARD_HEAD.parent.glob(
    SHARD_HEAD.name.replace("00001-of-", "*-of-")))) if SHARD_HEAD else 0
WANT_BYTES = (sum(q.stat().st_size for q in SHARD_HEAD.parent.glob(
    SHARD_HEAD.name.replace("00001-of-", "*-of-")))
    if SHARD_HEAD else int(MODEL_GB * 1e9))

# ═══════════════════════════════════════════════════════════════════════
# 1. Install Ollama
# ═══════════════════════════════════════════════════════════════════════
if shutil.which("ollama") is None:
    print("1/5 Installing Ollama ...")
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama install failed - check the output above.")
    print("    installed.")
else:
    print("1/5 Ollama already installed.")

# ═══════════════════════════════════════════════════════════════════════
# 2. Get the model onto local disk
# ═══════════════════════════════════════════════════════════════════════
LOCAL_OK, _ = store_report(SERVE_DIR, MODEL)
create_from = None
built_this_session = False

if LOCAL_OK:
    print(f"2/5 Local store has {MODEL}, blobs verified — skipping.")
elif DRIVE_STORE_OK:
    t0 = time.time()
    drive_gb = sum(q.stat().st_size for q in DRIVE_DIR.rglob('*') if q.is_file()) / 1e9
    print(f"2/5 Restoring store from Drive (~{drive_gb:.0f} GB) ...")
    # Only copy manifests and blobs, not loose .gguf files
    for sub in ("manifests", "blobs"):
        src = DRIVE_DIR / sub
        dst = SERVE_DIR / sub
        if src.exists():
            sh("cp", "-r", str(src), str(dst))
    print(f"    restored in {time.time() - t0:.0f}s.")
elif MERGED_IN_DRIVE and gguf_complete(MERGED_IN_DRIVE, WANT_BYTES):
    print(f"2/5 Using merged GGUF from Drive ({MERGED_IN_DRIVE.stat().st_size / 1e9:.1f} GB)")
    create_from = MERGED_IN_DRIVE
    built_this_session = True
elif SHARD_HEAD:
    merged = pathlib.Path('/content/merged.gguf')
    print(f"2/5 Merging {N_SHARDS} shards ...")
    LLAMA_BUILD = "b10701"
    if not shutil.which("llama-gguf-split"):
        url = (f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}"
               f"/llama-{LLAMA_BUILD}-bin-ubuntu-x64.tar.gz")
        !curl -sL -o /tmp/llama.tar.gz {url}
        !tar xzf /tmp/llama.tar.gz -C /tmp
        !cp /tmp/llama-{LLAMA_BUILD}/llama-gguf-split /usr/local/bin/
        !cp /tmp/llama-{LLAMA_BUILD}/*.so* /usr/local/lib/ 2>/dev/null
        !ldconfig
        !chmod +x /usr/local/bin/llama-gguf-split
    if merged.exists(): merged.unlink()
    t0 = time.time()
    sh("llama-gguf-split", "--merge", str(SHARD_HEAD), str(merged))
    if not gguf_complete(merged, WANT_BYTES):
        raise RuntimeError(f"Merge incomplete: {merged.stat().st_size / 1e9:.1f} GB")
    print(f"    merged {merged.stat().st_size / 1e9:.1f} GB in {time.time() - t0:.0f}s")
    # Save merged .gguf to Drive so we never merge again
    dst = DRIVE_DIR / f"{MODEL}.gguf"
    if not gguf_complete(dst, WANT_BYTES):
        print(f"    saving merged .gguf to Drive as {dst.name} ...")
        t0 = time.time()
        sh("cp", str(merged), str(dst))
        print(f"    saved in {time.time() - t0:.0f}s")
    merged.unlink()
    create_from = dst
    built_this_session = True
else:
    raise RuntimeError(
        f"Drive has no built store, no merged .gguf, and no shards.\n"
        f"Upload the BF16 shards into {DRIVE_DIR}")

# ═══════════════════════════════════════════════════════════════════════
# 3. Start the daemon
# ═══════════════════════════════════════════════════════════════════════
if daemon_up():
    print("3/5 Daemon already running.")
else:
    print(f"3/5 Starting daemon (log -> {OLLAMA_LOG}) ...")
    _log = open(OLLAMA_LOG, "ab")
    subprocess.Popen(["ollama", "serve"], stdout=_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        time.sleep(2)
        if daemon_up(): break
    else:
        raise RuntimeError("Ollama daemon never came up on :11434")
    print("    daemon up.")

# ═══════════════════════════════════════════════════════════════════════
# 4. Register if needed
# ═══════════════════════════════════════════════════════════════════════
if store_intact(SERVE_DIR, MODEL):
    print(f"4/5 {MODEL} already in store, blobs verified.")
elif create_from is not None:
    need = MODEL_GB * 2.2
    free = shutil.disk_usage(SERVE_DIR).free / 1e9
    print(f"4/5 ollama create {MODEL} ({free:.0f} GB free, need ~{need:.0f} GB) ...")
    if free < need:
        raise RuntimeError(f"Only {free:.0f} GB free; need ~{need:.0f} GB for ollama create.")
    mf = pathlib.Path('/content/Modelfile')
    mf.write_text(f"FROM {create_from}\n")
    t0 = time.time()
    sh_stream("ollama", "create", MODEL, "-f", str(mf))
    print(f"\n    created in {time.time() - t0:.0f}s")
else:
    raise RuntimeError(f"{MODEL} not in store and nothing to build from.")

# Verify it generates
print("    verifying model can generate ...")
ok, why = can_generate(MODEL)
if not ok:
    store_ok, problems = store_report(SERVE_DIR, MODEL)
    log_text = OLLAMA_LOG.read_text(errors="replace") if OLLAMA_LOG.exists() else ""
    detail = "\n".join(f"  {p}" for p in problems) if not store_ok else "store OK but generate failed"
    raise RuntimeError(f"{MODEL} will not generate: {why}\n{detail}\n\nLast log lines:\n{log_text[-1000:]}")
print("    verified: model generates.")

# ═══════════════════════════════════════════════════════════════════════
# 5. Save store to Drive (so future runs just restore)
# ═══════════════════════════════════════════════════════════════════════
if DRIVE_STORE_OK:
    print("5/5 Drive already has the store — nothing to save.")
elif built_this_session:
    print("5/5 Saving built store to Drive (one-time) ...")
    t0 = time.time()
    for sub in ("manifests", "blobs"):
        src = SERVE_DIR / sub
        dst = DRIVE_DIR / sub
        if src.exists():
            if dst.exists(): shutil.rmtree(dst)
            sh("cp", "-r", str(src), str(dst))
    drive_ok, drive_problems = store_report(DRIVE_DIR, MODEL)
    print(f"    saved in {time.time() - t0:.0f}s  "
          f"({'verified' if drive_ok else 'INCOMPLETE — ' + str(drive_problems[:2])})")
    if drive_ok:
        print("    Future runs will restore this store directly — no merge, no create.")
else:
    print("5/5 Model was restored from Drive — nothing new to save.")

print("\n" + "=" * 60)
!ollama list

## Step 5 — Preflight gate

Verifies the model is live, checks for tokenizer artifacts, and stamps provenance.

In [ ]:
import json, re, subprocess, urllib.error, urllib.request

ARTIFACTS = re.compile(r"\[UNK_BYTE_|▁")
fatal = []

tags = json.loads(urllib.request.urlopen(f"{HOST}/api/tags").read())["models"]
entry = next((m for m in tags if m["name"].split(":")[0] == MODEL), None)
if entry is None:
    fatal.append(f"{MODEL} is not in `ollama list` — Step 4 did not finish.")
else:
    digest = entry.get("digest", "?")
    quant = "?"
    try:
        info = json.loads(urllib.request.urlopen(
            urllib.request.Request(f"{HOST}/api/show",
                data=json.dumps({"name": MODEL}).encode(),
                headers={"Content-Type": "application/json"})).read())
        quant = {str(v): k for k, v in {
            "F32": 0, "F16": 1, "Q4_0": 2, "Q8_0": 7, "BF16": 32,
            "Q4_K_M": 15, "Q5_K_M": 17, "Q6_K": 18,
        }.items()}.get(str(info.get("model_info", {}).get("general.file_type", "?")), "?")
    except Exception:
        pass
    print(f"model:  {MODEL}")
    print(f"digest: {digest}")
    print(f"quant:  {quant}")

# One real token to detect byte-token wreckage
body = json.dumps({"model": MODEL, "prompt": "The patient was febrile at 39.2 °C.",
                    "stream": False, "format": "json",
                    "options": {"temperature": 0, "seed": 0, "num_predict": 64}}).encode()
req = urllib.request.Request(f"{HOST}/api/generate", data=body,
                             headers={"Content-Type": "application/json"})
try:
    resp = json.loads(urllib.request.urlopen(req, timeout=120).read())
    reply = resp.get("response", "")
    if ARTIFACTS.search(reply):
        fatal.append(f"Tokenizer artifacts in reply: {reply[:200]}")
        print(f"!!  REPLY: {reply[:200]}")
    else:
        print(f"preflight reply OK ({len(reply)} chars, no artifacts)")
except Exception as e:
    fatal.append(f"Preflight generation failed: {e}")

if fatal:
    raise RuntimeError("Preflight FAILED:\n  " + "\n  ".join(fatal))
print("\nPreflight passed — the model is live and generating clean text.")

## Step 6 — Run the extraction

Configure, then run. The result lands at `results/stage_{PROMPT_STAGE}.json`.

In [ ]:
import os, pathlib, subprocess, time

# ═══════════════════════════════════════════════════════════════════════
# Configuration — edit these
# ═══════════════════════════════════════════════════════════════════════
PROMPT_STAGE = "3"          # prompt stage (0, 1, 2a, 2b, 3)
NOTES = 20                  # number of notes to run
REPEAT = 2                  # repeats for run-to-run consistency
MEASURE_CONSISTENCY = True  # True = sequential (consistency meaningful)

CONCURRENCY = 1 if MEASURE_CONSISTENCY else NUM_PARALLEL
OUT_PATH = f"results/stage_{PROMPT_STAGE}.json"
pathlib.Path("results").mkdir(exist_ok=True)

print(f"stage={PROMPT_STAGE}  notes={NOTES}  repeat={REPEAT}  concurrency={CONCURRENCY}")
print(f"output -> {OUT_PATH}")

t0 = time.time()
cmd = (f"python scripts/experiments/medgemma_extraction.py"
       f" --tier full --model {MODEL} --backend ollama"
       f" --cohort scd_primary --stratify --holdout-frac 0.25"
       f" --notes {NOTES} --repeat {REPEAT} --concurrency {CONCURRENCY}"
       f" --prompt-stage {PROMPT_STAGE} --out {OUT_PATH}")
!{cmd}

if not os.path.exists(OUT_PATH):
    raise SystemExit(f"No result file at {OUT_PATH} — the run failed.")
print(f"\nDone in {(time.time() - t0) / 60:.1f} min")

## Step 7 — Results

Grounding, automated metrics, per-outcome breakdown, seeded vs holdout.

In [ ]:
import json, os
import pandas as pd

RESULT_PATH = f"results/stage_{PROMPT_STAGE}.json"
assert os.path.exists(RESULT_PATH), f"No result at {RESULT_PATH}"

with open(RESULT_PATH, encoding="utf-8") as f:
    data = json.load(f)

prov, prof = data["provenance"], data["profiling"]
met, runs = data["automated_metrics"], data["runs"]

print("=" * 74)
print(f"MedGemma extraction report   stage={prov.get('prompt_stage', '?')}  "
      f"tier={prov.get('tier', '?')}  cohort={prov.get('cohort', '?')}")
print(f"weights={prov.get('weights')}   served_as={prov.get('served_as')}   "
      f"backend={prov.get('backend')}")
print(f"digest={prov.get('model_digest')}   quant={prov.get('quant')}   "
      f"{prov.get('timestamp')}")
RUN_ID = prov.get("run_id", prov.get("timestamp"))
print(f"run_id={RUN_ID}")
print("=" * 74)

# Cohort
sel = data.get("selection", {})
print(f"\nCOHORT   {prov.get('cohort')}   {sel.get('selected', '?')}/{sel.get('pool', '?')} "
      f"selected notes are SCD-primary")

# Grounding
print(f"\nGROUNDING   ({met['proposed']} proposed findings)")
print(f"  null placeholders    {met.get('null_placeholder', 0):3d}     "
      f"{met.get('null_placeholder_pct', 0):.1f}% of proposed")
print(f"  quote verified       {met.get('accepted', 0):3d}     "
      f"{met.get('quote_verified_pct_of_quoted', met.get('accepted', 0) / max(met.get('proposed', 1) - met.get('null_placeholder', 0), 1) * 100):.1f}% of quoted")
print(f"  quote not in note    {met.get('hallucinated', met.get('proposed', 0) - met.get('accepted', 0) - met.get('null_placeholder', 0)):3d}     "
      f"{met.get('hallucinated_pct_of_quoted', 0):.1f}% of quoted")

# Automated metrics table
rows = [
    ("Quote-verified % (of quoted)", f"{100 - met.get('hallucinated_pct_of_quoted', 0):.1f}%",
     "≥95 / 85-95 / <85"),
    ("Run-to-run consistency", f"{met.get('run_to_run_consistency_pct', 0):.1f}%",
     "≥98 / 90-98 / <90"),
    ("Invalid-value rate", f"{met.get('invalid_value_pct', 0):.1f}%", "≤2 / 2-10 / >10"),
    ("Null-placeholder rate", f"{met.get('null_placeholder_pct', 0):.1f}%", "≤5"),
]
mdf = pd.DataFrame(rows, columns=["P11 automated metric", "Value", "Thresholds"])
display(mdf)

# Cost
cost_rows = [
    ("Notes × outcomes", f"{prof.get('notes', '?')} × {prof.get('outcomes_per_note', '?')}"),
    ("Wall clock", f"{prof.get('total_wall_clock_sec', 0):.0f} s"),
    ("Per note", f"{prof.get('total_wall_clock_sec', 0) / max(prof.get('notes', 1), 1):.1f} s"),
]
display(pd.DataFrame(cost_rows, columns=["Cost", "Value"]))

# Grade status
g = data.get("grade_status", {})
gs = pd.DataFrame([
    {"SCOGS status": k, "Note-outcome pairs": v,
     "Share": f"{v / sum(g.values()) * 100:.1f}%"}
    for k, v in sorted(g.items(), key=lambda x: -x[1])
])
display(gs)

# Per outcome
recs = data["detailed_records"]
per = {}
for rec in recs:
    for num, o in rec["outcomes"].items():
        st = o["grade_result"]["status"]
        per.setdefault(num, {"outcome": f"{num} {o['outcome_name']}"})
        per[num][st] = per[num].get(st, 0) + 1
po = pd.DataFrame(per.values()).fillna(0).astype({c: int for c in
    ["graded", "grade_set", "cannot_grade", "absent", "refuted"] if c != "outcome"}, errors="ignore")
print("\nPER OUTCOME")
display(po)

# Seeded vs holdout
sel_rows = []
for label in sorted(set(rec.get("selection", "?").split(":")[0] for rec in recs)):
    subset = [rec for rec in recs if rec.get("selection", "").startswith(label)]
    pairs = sum(len(rec["outcomes"]) for rec in subset)
    counts = {}
    for rec in subset:
        for o in rec["outcomes"].values():
            s = o["grade_result"]["status"]
            counts[s] = counts.get(s, 0) + 1
    row = {"selection": label, "pairs": pairs}
    for s in ["graded", "grade_set", "cannot_grade", "absent", "refuted"]:
        c = counts.get(s, 0)
        row[s] = f"{c} ({c/pairs*100:.0f}%)" if pairs else "0"
    sel_rows.append(row)
print("\nSEEDED vs HOLDOUT")
display(pd.DataFrame(sel_rows))

# Feature frequency
feats = {}
for rec in recs:
    for o in rec["outcomes"].values():
        for feat, info in o.get("extracted_features", {}).items():
            if info.get("status") == "accepted":
                feats[feat] = feats.get(feat, 0) + 1
feat_df = pd.DataFrame([{"Feature": k, "Times accepted": v}
    for k, v in sorted(feats.items(), key=lambda x: -x[1])])
display(feat_df)

## Step 8 — Hand-check worksheet

Generates the precision review sheet. Fill in `supports_value` as y/n.

In [ ]:
import sys
sys.path.insert(0, "scripts/experiments")

HANDCHECK_PATH = f"results/handcheck_stage{PROMPT_STAGE}.csv"
CONFLICT_PATH = f"results/conflicts_stage{PROMPT_STAGE}.csv"

# Build the findings DataFrame from the detailed records
import pandas as pd
rows = []
for rec in data["detailed_records"]:
    for num, o in rec["outcomes"].items():
        for feat, info in o.get("extracted_features", {}).items():
            if info.get("status") == "accepted":
                rows.append({
                    "run_id": RUN_ID, "uid": rec["patient_uid"],
                    "outcome": f"{num} {o['outcome_name']}",
                    "feature": feat, "value": info["value"],
                    "quote": info.get("quote", ""),
                    "unit_note": info.get("unit_note", ""),
                    "status": o["grade_result"]["status"],
                    "grade": o["grade_result"].get("grade", ""),
                })
df = pd.DataFrame(rows)

# Conflicts (multi-value features where values disagreed)
conf_rows = []
for rec in data["detailed_records"]:
    for num, o in rec["outcomes"].items():
        for feat, info in o.get("extracted_features", {}).items():
            if info.get("status") == "conflict":
                conf_rows.append({
                    "run_id": RUN_ID, "uid": rec["patient_uid"],
                    "outcome": f"{num} {o['outcome_name']}",
                    "feature": feat, "values": str(info.get("candidates", [])),
                    "quotes": str(info.get("candidate_quotes", [])),
                })
conf = pd.DataFrame(conf_rows)

if len(df):
    work = df.copy()
    work.insert(len(work.columns), "supports_value", "")
    work.insert(len(work.columns), "reviewer_note", "")
    work = work.sample(n=min(100, len(work)), random_state=0)
    work.to_csv(HANDCHECK_PATH, index=False)
    print(f"Wrote {len(work)} rows to {HANDCHECK_PATH}   (run_id {RUN_ID})")
    print("\nOpen it, mark supports_value as y/n, then compute:")
    print("   precision = y / (y + n)")
else:
    print("Nothing accepted to hand-check.")

if len(conf):
    conf.to_csv(CONFLICT_PATH, index=False)
    print(f"\nWrote {len(conf)} conflict rows to {CONFLICT_PATH}")

## Step 9 — Absence audit

Generates the false-negative review sheet. Fill in `truly_absent` as y/n.

In [ ]:
from medgemma_extraction import feature_brief, prompt_features, stage

ABSENCE_PATH = f"results/absence_audit_stage{PROMPT_STAGE}.csv"
REFUTED_PATH = f"results/refuted_audit_stage{PROMPT_STAGE}.csv"

_ST = stage(prov.get("prompt_stage", "0"))

def what_would_make_it_present(num):
    return " | ".join(feature_brief(n, num, _ST) for n in prompt_features(num, _ST))

abs_rows, ref_rows = [], []
for rec in data["detailed_records"]:
    for num, o in rec["outcomes"].items():
        gr = o["grade_result"]
        if gr["status"] not in ("absent", "refuted"):
            continue
        row = {
            "run_id": RUN_ID,
            "uid": rec["patient_uid"],
            "selection": rec.get("selection", ""),
            "outcome": num,
            "outcome_name": o["outcome_name"],
            "model_said_present": o.get("present"),
            "truly_absent": "",
            "reviewer_note": "",
            "look_for": what_would_make_it_present(num),
            "title": rec.get("title", ""),
            "note_text": rec["patient_note"],
        }
        if gr["status"] == "absent":
            abs_rows.append(row)
        else:
            row["extracted_features"] = json.dumps(o["extracted_features"])
            row["rule_reason"] = gr.get("reason") or ""
            ref_rows.append(row)

adf = pd.DataFrame(abs_rows)
if len(adf):
    sample = adf.sample(n=min(50, len(adf)), random_state=0).sort_values(["uid", "outcome"])
    sample.to_csv(ABSENCE_PATH, index=False)
    print(f"{len(adf)} absent pairs; wrote {len(sample)} to {ABSENCE_PATH}")
    print(f"   (run_id {RUN_ID})")
    print("\n  false-negative rate = n / (y + n)")
    display(sample.groupby("outcome_name").size().rename("absent pairs").to_frame())
else:
    print("Nothing marked absent.")

rdf = pd.DataFrame(ref_rows)
if len(rdf):
    rdf.to_csv(REFUTED_PATH, index=False)
    print(f"\n{len(rdf)} refuted pairs -> {REFUTED_PATH}")
    display(rdf[["uid", "outcome", "outcome_name", "extracted_features", "rule_reason"]])

## Step 10 — Save artifacts

Drive first, browser download second. Everything under `/content` is deleted when
the VM stops.

In [ ]:
import glob, os, pathlib, shutil, zipfile

# Gather artifacts
artifacts  = sorted(glob.glob(f"results/stage_{PROMPT_STAGE}.json"))
artifacts += sorted(glob.glob(f"results/*_stage{PROMPT_STAGE}.csv"))

assert artifacts, (
    f"Nothing to save. Steps 6-9 write results/stage_{PROMPT_STAGE}.json and "
    f"results/*_stage{PROMPT_STAGE}.csv; both globs are empty.")

result_json = f"results/stage_{PROMPT_STAGE}.json"
assert result_json in artifacts, f"{result_json} is missing."

for name, why in ((f"results/conflicts_stage{PROMPT_STAGE}.csv",
                   "no findings were withheld as conflicting"),
                  (f"results/refuted_audit_stage{PROMPT_STAGE}.csv",
                   "no pair was refuted by the decision tables")):
    if name not in artifacts:
        print(f"note: {os.path.basename(name)} not written — {why}.")

print(f"\n{len(artifacts)} artifact(s) for run_id {RUN_ID}:")
for p in artifacts:
    print(f"   {os.path.getsize(p) / 1e6:7.2f} MB  {p}")

# Save to Drive
if os.path.isdir("/content/drive/MyDrive"):
    DRIVE_OUT = pathlib.Path("/content/drive/MyDrive/scogs_results") / (RUN_ID or prov["timestamp"])
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    for p in artifacts:
        shutil.copy2(p, DRIVE_OUT / os.path.basename(p))
    (DRIVE_OUT / "MANIFEST.txt").write_text(
        f"run_id     {RUN_ID}\n"
        f"stage      {prov.get('prompt_stage')}\n"
        f"cohort     {prov.get('cohort')}\n"
        f"weights    {prov['weights']}  served_as {prov['served_as']}\n"
        f"digest     {prov.get('model_digest')}   quant {prov.get('quant')}\n"
        f"timestamp  {prov['timestamp']}\n\n"
        + "\n".join(sorted(os.path.basename(p) for p in artifacts)) + "\n")
    print(f"\nSaved to Drive -> {DRIVE_OUT}")
else:
    print("\n!! NOT saved to Drive — Drive not mounted.")

# Browser download as zip
ZIP = f"/content/scogs_{RUN_ID or 'run'}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        z.write(p, os.path.join(RUN_ID or "run", os.path.basename(p)))
print(f"Zip {os.path.getsize(ZIP) / 1e6:.1f} MB -> {ZIP}")

from google.colab import files
files.download(ZIP)

print("\nAll files carry the same run_id. Keep them together.")